# Kings HUDL game analyzer

Press the Play button on each box below, one at a time, top to bottom. Wait for each one to finish before pressing the next.

Before you start: click **Runtime**, then **Change runtime type**, pick **T4 GPU**, and press Save. Without a graphics card this is very slow.

**If Drive will not connect** ("credential propagation was unsuccessful"): sign out of other Google accounts in this Chrome, allow third party cookies for colab.research.google.com, then click Runtime, then Disconnect and delete runtime, and try again. Or skip step 2: click the folder icon on the left, drag your game video into it (it can take a while), and in step 3 type /content/ plus the file name.


In [ ]:
#@title 1. Get the newest program
import os, subprocess
if not os.path.exists('/content/kings-hudl'):
    subprocess.run(['git','clone','-q','https://github.com/rlahay-cloud/kings-hudl','/content/kings-hudl'],check=True)
else:
    subprocess.run(['git','-C','/content/kings-hudl','pull','-q'],check=True)
!pip -q install ultralytics lapx scikit-learn scipy easyocr
import torch
print('Graphics card:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE. Change the runtime type to T4 GPU, or this will take days.')

In [ ]:
#@title 2. Connect to Google Drive
#@markdown If this step fails, skip it and upload the video straight into Colab instead (see the notes at the top).
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print('Drive did not connect:', e)
    print('You can skip this step and upload the video into Colab with the folder icon on the left.')


In [ ]:
#@title 3. Pick the game
#@markdown If Drive is connected, type the path of the game file inside your Drive, starting after "My Drive/".
#@markdown If you uploaded the file straight into Colab (folder icon on the left), type its full name, like /content/game.mov
GAME_FILE = "KingsHUDL games/2026-10-24_Rapids.mov"  #@param {type:"string"}
#@markdown Leave these at 0 to do the whole game. To do one period, type its start and end in seconds.
START_SECONDS = 0  #@param {type:"number"}
END_SECONDS = 0  #@param {type:"number"}
import os, shutil
src = GAME_FILE if GAME_FILE.startswith('/') else '/content/drive/MyDrive/' + GAME_FILE
assert os.path.exists(src), 'I cannot find that file. Check the name and the folder.'
name = os.path.splitext(os.path.basename(GAME_FILE))[0]
drive_ok = os.path.isdir('/content/drive/MyDrive')
OUT = ('/content/drive/MyDrive/KingsHUDL games/analysis_' if drive_ok else '/content/analysis_') + name
os.makedirs(OUT, exist_ok=True)
if src.startswith('/content/drive/'):
    local = '/content/game' + os.path.splitext(src)[1]
    if not os.path.exists(local):
        print('Copying the video to the fast disk, this takes a few minutes...')
        shutil.copy(src, local)
else:
    local = src
print('Ready:', local, '->', OUT, '' if drive_ok else '(Drive is not connected, so results are only downloaded, not saved in Drive)')


In [ ]:
#@title 4. Analyze (this is the long one)
#@markdown It saves its work every picture. If it stops, press Play on this box again and it carries on.
import sys
sys.path.insert(0, '/content/kings-hudl/analyzer')
import kings_analyzer as ka
ka.detect(local, OUT, '/content/kings-hudl/analyzer/rinks/ingredion_rink1.json',
          step=0.5, start=START_SECONDS, end=(END_SECONDS or None), imgsz=1600, device=0)

In [ ]:
#@title 5. Read the scoreboard (clock, goals, penalty boxes)
#@markdown This reads the real scoreboard in the picture once a second. It is quick compared with step 4.
import scoreboard
scoreboard.read_series(local, '/content/kings-hudl/analyzer/rinks/ingredion_rink1.json', OUT, step=1.0,
                       start=START_SECONDS, end=(END_SECONDS or None))

In [ ]:
#@title 6. Make the review page
#@markdown Leave "try numbers" on to let the program guess jersey numbers. The guesses are often wrong, so they only show as small hints.
TRY_NUMBERS = True  #@param {type:"boolean"}
page = ka.build(local, OUT, '/content/kings-hudl/analyzer/rinks/ingredion_rink1.json', read_nums=TRY_NUMBERS)
print('Done. Open this file from your Drive (right click, Open with, a web browser):')
print(page)
from google.colab import files
files.download(page)